# Ensemble vs. single model — comparison report

**What this notebook compares.** Two dispatch orderings over the same 1,164
towers, the same synthetic labels and the same state-grouped folds:

| policy | what produces the ordering |
|---|---|
| **single model** | `model/maintenance_need.py` — LightGBM over 12 features, out-of-fold probability `risk` |
| **ensemble** | `model/ensemble.py::blend_priority` — a rank blend of that probability with the one-sided condition score over the 30-day telemetry block, at `CONDITION_BLEND_WEIGHT` |

Two reference rows are carried through every table so the numbers are readable
rather than merely large: the retired **noisy-OR AHP index** (the baseline the
supervised model replaced) and the generator's own **latent intensity**
(`lambda_true`, an oracle — the ceiling, and the leak check).

**Three things this notebook does not do.**

1. It does not retrain. It re-derives the out-of-fold predictions with the same
   helper the training notebook uses, so the figures here and in
   `maintenance_need_report.json` come from one procedure, not two.
2. It does not fit anything to the comparison. `CONDITION_BLEND_WEIGHT` is
   imported, never chosen here; the weight was swept on seeds 0–4 and verified
   on 5–9 in `notebooks/maintenance_need.ipynb`, and re-picking it against these
   plots would be selecting on the test set.
3. It does not claim real-world accuracy. **The labels are synthetic.** No
   operator or MCMC maintenance history exists for this prototype. Every figure
   validates the evaluation pipeline; none of them validates field performance.

**The comparison rule that governs the whole report.** Any policy that dispatches
more towers must be scored against a *cut* dispatching the same number — never
against the untouched top-10%. A gate on `novelty` once shipped here and was
removed by exactly that comparison. Every confusion matrix below is at a
**matched budget**.

In [ ]:
import json
import sys
from pathlib import Path

BACKEND = Path.cwd().parent / "src" / "backend"
sys.path.insert(0, str(BACKEND))
sys.path.insert(0, str(BACKEND / "data"))

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from sklearn.metrics import precision_recall_curve, roc_auc_score as roc_auc, roc_curve
from sklearn.model_selection import GroupKFold

# Shared constants are IMPORTED, never redefined. A notebook that declares its
# own FEATURES or its own blend weight is reporting on a different system from
# the one the API serves.
from model.maintenance_need import (
    CATEGORICAL_FEATURES, EARLY_STOPPING_ROUNDS, FEATURES, NUM_BOOST_ROUND,
    PARAMS, TOP_K, build_matrix,
)
from model.ensemble import CHANGE_BLEND_WEIGHT, CONDITION_BLEND_WEIGHT, blend_priority
from model.change import change_scores
from model.novelty import TELEMETRY_COLUMNS, condition_scores
from model.flood_eval import ranking_metrics, top_k_confusion
from model.risk_index import AGE_WEIGHT, AHP_FACTORS, ahp_weights, memberships, noisy_or

DATA = BACKEND.parent.parent / "data" / "malaysia"

plt.rcParams.update({"figure.dpi": 120, "font.size": 9, "axes.grid": True,
                     "grid.alpha": 0.25, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.autolayout": True})

# One colour per policy, used by every figure in the report so a reader learns
# the mapping once. Grey is reserved for the two reference rows: they are
# context, not candidates.
COLOR = {"single model": "#2563eb", "ensemble": "#c2410c",
         "AHP index": "#94a3b8", "oracle": "#475569"}
STYLE = {"single model": "-", "ensemble": "-", "AHP index": "--", "oracle": ":"}

print("blend weight:", CONDITION_BLEND_WEIGHT, "| change weight:", CHANGE_BLEND_WEIGHT)
print("features:", len(FEATURES), "| operating point: top-", int(TOP_K * 100), "%", sep="")

## 1. Load — one population, one label, one grouping

The label is `needed_corrective_maintenance`: *a corrective work order was
raised*. It is **not** an outage, a fault, or a probability of one. The manifest
is asserted to declare itself synthetic, so a run against a differently-sourced
label fails here rather than quietly producing a credible-looking report.

In [ ]:
towers = pd.read_csv(DATA / "tower_feature_table.csv")
land = pd.read_csv(DATA / "land_features.csv")
labels = pd.read_csv(DATA / "maintenance_labels.csv")
manifest = json.loads((DATA / "maintenance_manifest.json").read_text())

df = (towers.merge(land, on="tower_id")
            .merge(labels.drop(columns=["state"]), on="tower_id"))
df["radio"] = df["radio"].fillna("UNKNOWN")
df["state"] = df["state"].fillna("unassigned").replace("", "unassigned")

y = df["needed_corrective_maintenance"].to_numpy()
groups = df["state"].to_numpy()
X = build_matrix(df)
base_rate = float(y.mean())

assert manifest["synthetic"] is True, "labels must declare themselves synthetic"
print(f"{len(df)} towers | {int(y.sum())} positives | base rate {base_rate:.4f}")
print(f"{df.state.nunique()} ADM1 groups | window {manifest['window'][0]} to {manifest['window'][1]}")
print(f"generator v{manifest.get('generator_version', '?')}")

## 2. The two orderings

**Why GroupKFold on ADM1 state, not a random split.** Towers a few kilometres
apart share a catchment, a contractor and a weather history, so a random split
scores near-duplicates and reports a number that will not survive a new state.
Only the *fitted* model needs held-out predictions; the condition score is
label-free, so it describes the same population the serving bands are cut over —
which is exactly what serving does.

**Why the ensemble blends ranks rather than values.** `risk` is a calibrated
probability and `condition` is already a percentile. Averaging them directly
would let the model's scale silently decide the weight. Ranking both first makes
`CONDITION_BLEND_WEIGHT` mean what it says.

**Why telemetry is not simply a feature.** The counters cover 30 days; the label
covers 36 months. There is no aligned `(telemetry, outcome)` pair in this dataset
to fit, so the supervised model cannot use it — and an unsupervised read needs no
labels at all. That asymmetry is the entire reason a second member earns a place
here. A tower with no telemetry keeps its bare model rank; missing monitoring is
not evidence of a healthy site.

In [ ]:
def out_of_fold(X, y, groups, splitter, params=PARAMS):
    # Pooled out-of-fold probabilities: every row is scored by a fit that never
    # saw its group. An inner 85/15 split drives early stopping, so no fold
    # chooses its own round count against the rows it is then scored on.
    out = np.full(len(y), np.nan)
    for train_idx, test_idx in splitter.split(X, y, groups):
        cut = int(len(train_idx) * 0.85)
        fit_idx, val_idx = train_idx[:cut], train_idx[cut:]
        booster = lgb.train(
            params,
            lgb.Dataset(X.iloc[fit_idx], y[fit_idx], categorical_feature=CATEGORICAL_FEATURES),
            num_boost_round=NUM_BOOST_ROUND,
            valid_sets=[lgb.Dataset(X.iloc[val_idx], y[val_idx],
                                    categorical_feature=CATEGORICAL_FEATURES)],
            callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)],
        )
        out[test_idx] = booster.predict(X.iloc[test_idx])
    return out


n_splits = min(5, df.state.nunique())
grouped = GroupKFold(n_splits=n_splits)
oof = out_of_fold(X, y, groups, grouped)
assert not np.isnan(oof).any(), "every tower must be scored out-of-fold"

telemetry = pd.read_csv(DATA / "site_telemetry.csv")
condition = condition_scores(df.tower_id.to_numpy(), telemetry)
change = change_scores(df.tower_id.to_numpy())

# The adapter's own blend, so this report and the API cannot disagree about the
# ordering that decides dispatch.
priority = blend_priority(oof, condition, change)

# Two reference rows. The index is the baseline the supervised model replaced;
# the oracle is the generator's latent intensity — the ceiling, and the leak check.
p_index = memberships(df).drop(columns=["lightning"], errors="ignore")
w0, _ = ahp_weights()
index_weights = dict(zip(AHP_FACTORS, w0 / w0.max()))
index_weights["age"] = AGE_WEIGHT
index_risk = noisy_or(p_index, {c: index_weights[c] for c in p_index.columns})
oracle = df["lambda_true"].to_numpy()

POLICIES = {"single model": oof, "ensemble": priority,
            "AHP index": index_risk, "oracle": oracle}

no_telemetry = int(np.isnan(condition).sum())
print(f"{len(df) - no_telemetry} towers carry telemetry; {no_telemetry} keep a bare model rank")
print(f"raw telemetry ROC against the label: "
      f"{roc_auc(y, np.log1p(telemetry.set_index('tower_id').reindex(df.tower_id)[TELEMETRY_COLUMNS]).mean(axis=1)):.4f}")
agreement = pd.Series(oof).corr(pd.Series(condition), method="spearman")
print(f"spearman(model, condition) = {agreement:+.4f}")
print("The two agree substantially, which is expected — both read the same underlying")
print("site hazard. Agreement alone is neither evidence for nor against the blend: what")
print("decides it is whether condition beats the model INSIDE the contested band (§7).")

## 3. Ranking quality — the whole ordering, before any cut

ROC-AUC and PR-AUC score the *ordering*, independent of how many crews exist.
**PR-AUC is the one to read**: at a 21% base rate ROC is dominated by the
negatives, and the product question is the purity of a small dispatched slice.

`priority` is a percentile rank, not a probability. It must never be served as
one — which is why the adapter emits both it and `risk`.

In [ ]:
rank_rows = []
for name, score in POLICIES.items():
    k = ranking_metrics(y, score)
    rank_rows.append({"policy": name, "roc_auc": k["roc_auc"], "pr_auc": k["pr_auc"]})
ranking_table = pd.DataFrame(rank_rows).set_index("policy")

delta_pr = ranking_table.loc["ensemble", "pr_auc"] - ranking_table.loc["single model", "pr_auc"]
delta_roc = ranking_table.loc["ensemble", "roc_auc"] - ranking_table.loc["single model", "roc_auc"]

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))

for name, score in POLICIES.items():
    fpr, tpr, _ = roc_curve(y, score)
    ax[0].plot(fpr, tpr, STYLE[name], color=COLOR[name], lw=1.6,
               label=f"{name}  {ranking_metrics(y, score)['roc_auc']:.4f}")
    prec, rec, _ = precision_recall_curve(y, score)
    ax[1].plot(rec, prec, STYLE[name], color=COLOR[name], lw=1.6,
               label=f"{name}  {ranking_metrics(y, score)['pr_auc']:.4f}")

ax[0].plot([0, 1], [0, 1], color="#cbd5e1", lw=1)
ax[0].set(xlabel="false positive rate", ylabel="true positive rate", title="ROC")
ax[1].axhline(base_rate, color="#cbd5e1", lw=1)
ax[1].annotate(f"base rate {base_rate:.3f}", (0.02, base_rate), fontsize=8,
               va="bottom", color="#64748b")
ax[1].set(xlabel="recall", ylabel="precision", title="Precision-recall (read this one)")
for a in ax:
    a.legend(fontsize=8, loc="lower left" if a is ax[0] else "upper right")
fig.suptitle("Ranking quality over the full population", y=1.04, fontsize=10)
plt.show()

display(ranking_table)
print(f"ensemble minus single model:  PR-AUC {delta_pr:+.4f}   ROC-AUC {delta_roc:+.4f}")
print(f"gap from ensemble to oracle:  PR-AUC "
      f"{ranking_table.loc['oracle', 'pr_auc'] - ranking_table.loc['ensemble', 'pr_auc']:+.4f}"
      "   (a policy landing ON the oracle would mean the generator leaked)")

## 4. Accuracy, precision, recall, F1 — at the matched operating point

Crew capacity decides how many towers get dispatched, not a probability
threshold. So both policies are cut at the **same budget**: the top 10% of the
estate, `TOP_K`. Equal budget is what makes precision, recall and F1 comparable
at all — a policy dispatching more towers buys recall for free.

**Accuracy is reported and flagged.** At a 21% base rate, a policy that dispatches
nothing scores 0.79 on it. It is here because it is always asked for, not because
it separates these two.

In [ ]:
budget = int(round(TOP_K * len(y)))

def confusion_at_budget(score, budget):
    # Top-`budget` selection, then the standard cells. Ties broken stably, so a
    # flat region of either score cannot make the comparison order-dependent.
    selected = np.zeros(len(y), bool)
    selected[np.argsort(-score, kind="stable")[:budget]] = True
    tp = int((selected & (y == 1)).sum())
    fp = int((selected & (y == 0)).sum())
    fn = int((~selected & (y == 1)).sum())
    tn = int((~selected & (y == 0)).sum())
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    specificity = tn / max(1, tn + fp)
    return {
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "precision": round(precision, 4), "recall": round(recall, 4),
        "f1": round(2 * precision * recall / max(1e-12, precision + recall), 4),
        "accuracy": round((tp + tn) / len(y), 4),
        "balanced_accuracy": round((recall + specificity) / 2, 4),
        "lift_over_random": round(precision / base_rate, 3),
    }

matched = pd.DataFrame({name: confusion_at_budget(s, budget)
                        for name, s in POLICIES.items()}).T
matched.index.name = f"policy (all dispatch {budget} towers)"
display(matched)

METRICS = ["precision", "recall", "f1", "accuracy", "balanced_accuracy"]
compare = ["single model", "ensemble"]
x = np.arange(len(METRICS))
width = 0.36

fig, ax = plt.subplots(figsize=(7.4, 3.6))
for i, name in enumerate(compare):
    vals = [matched.loc[name, m] for m in METRICS]
    bars = ax.bar(x + (i - 0.5) * width, vals, width, color=COLOR[name], label=name)
    ax.bar_label(bars, fmt="%.3f", fontsize=7.5, padding=2)
# The two reference rows as marks rather than bars: they are context, and a bar
# invites a reader to compare them as candidates at this budget.
for name in ("AHP index", "oracle"):
    ax.scatter(x, [matched.loc[name, m] for m in METRICS], marker="_", s=260,
               color=COLOR[name], zorder=5, label=name)
ax.axhline(base_rate, color="#cbd5e1", lw=1)
ax.annotate("base rate", (len(METRICS) - 0.45, base_rate), fontsize=7.5,
            va="bottom", ha="right", color="#64748b")
ax.set(xticks=x, ylim=(0, 1.05), ylabel="score",
       title=f"Matched budget: both policies dispatch {budget} towers")
ax.set_xticklabels(METRICS, fontsize=8.5)
ax.legend(fontsize=8, ncol=4, loc="lower center", bbox_to_anchor=(0.5, -0.32))
plt.show()

d = matched.loc["ensemble"] - matched.loc["single model"]
print(f"ensemble minus single model at budget {budget}:  "
      f"F1 {d['f1']:+.4f}   TP {int(d['TP']):+d}   FP {int(d['FP']):+d}   "
      f"precision {d['precision']:+.4f}   recall {d['recall']:+.4f}")
print("Accuracy barely moves and that is expected: 90% of the estate is untouched "
      "by either policy, so the shared TN block dominates it.")

### The confusion matrices side by side

Same budget, so every tower the ensemble gains is a tower the single model gave
up. The interesting cells are TP and FP; TN is large and shared.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.6, 3.2))
for ax_, name in zip(axes, compare):
    r = matched.loc[name]
    cells = np.array([[r["TN"], r["FP"]], [r["FN"], r["TP"]]], dtype=float)
    ax_.imshow(np.log1p(cells), cmap="Blues", alpha=0.45)
    for (i, j), v in np.ndenumerate(cells):
        ax_.text(j, i, f"{int(v)}", ha="center", va="center", fontsize=12,
                 color="#0f172a")
    ax_.set(xticks=[0, 1], yticks=[0, 1], title=f"{name}\nF1 {r['f1']:.4f}",
            xlabel="not dispatched / dispatched", ylabel="no work / work needed")
    ax_.set_xticklabels(["not disp.", "dispatched"], fontsize=8)
    ax_.set_yticklabels(["no work", "work needed"], fontsize=8)
    ax_.grid(False)
plt.show()

## 5. Does the gain survive a different budget?

A single operating point can flatter either policy. Crew capacity is a planning
decision, not a constant, so the honest question is whether the ensemble leads
*across* the budgets a planner might actually run — not only at exactly 10%.

The shaded band marks where the ensemble is ahead.

In [ ]:
budgets = np.arange(int(0.03 * len(y)), int(0.31 * len(y)) + 1, 4)
curve = pd.DataFrame([
    {"budget": b, "policy": name, **confusion_at_budget(s, int(b))}
    for b in budgets for name, s in POLICIES.items()
])

fig, ax = plt.subplots(1, 3, figsize=(12, 3.5))
for metric, a in zip(["f1", "precision", "recall"], ax):
    for name in POLICIES:
        sub = curve[curve.policy == name]
        a.plot(sub.budget / len(y) * 100, sub[metric], STYLE[name],
               color=COLOR[name], lw=1.6, label=name)
    a.axvline(TOP_K * 100, color="#cbd5e1", lw=1)
    a.set(xlabel="dispatched (% of estate)", ylabel=metric, title=metric)
ax[0].annotate("served\ntop-10%", (TOP_K * 100 + 0.6, ax[0].get_ylim()[0] + 0.02),
               fontsize=7.5, color="#64748b")

wide = curve.pivot_table(index="budget", columns="policy", values="f1")
ax[0].fill_between(wide.index / len(y) * 100, wide["single model"], wide["ensemble"],
                   where=wide["ensemble"] >= wide["single model"],
                   color=COLOR["ensemble"], alpha=0.12, lw=0)
ax[2].legend(fontsize=8, loc="lower right")
fig.suptitle("Metrics against dispatch budget — one operating point can flatter either policy",
             y=1.05, fontsize=10)
plt.show()

wins = int((wide["ensemble"] > wide["single model"]).sum())
print(f"ensemble leads on F1 at {wins} of {len(wide)} budgets from 3% to 30% of the estate")
print(f"mean F1 gap over that range: "
      f"{(wide['ensemble'] - wide['single model']).mean():+.4f}")

## 6. Where the gain comes from: the swap

Matched budget means the two lists differ by a swap, not a widening. This is the
most diagnostic view in the report — if the towers the ensemble swaps in hit at
roughly the base rate, the blend is shuffling noise, and the F1 gain above is
sampling luck rather than information.

In [ ]:
def top_mask(score, budget):
    m = np.zeros(len(y), bool)
    m[np.argsort(-score, kind="stable")[:budget]] = True
    return m

ens_mask = top_mask(priority, budget)
mod_mask = top_mask(oof, budget)
swapped_in = ens_mask & ~mod_mask       # ensemble dispatches, model did not
swapped_out = mod_mask & ~ens_mask      # model dispatched, ensemble dropped

hit_in = float(y[swapped_in].mean()) if swapped_in.any() else float("nan")
hit_out = float(y[swapped_out].mean()) if swapped_out.any() else float("nan")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))

bars = ax[0].bar(["swapped IN\n(ensemble adds)", "swapped OUT\n(ensemble drops)",
                  "below the model's cut"],
                 [hit_in, hit_out, float(y[~mod_mask].mean())],
                 color=[COLOR["ensemble"], COLOR["single model"], "#cbd5e1"])
ax[0].bar_label(bars, fmt="%.3f", fontsize=8)
ax[0].axhline(base_rate, color="#64748b", lw=1, ls="--")
ax[0].annotate(f"base rate {base_rate:.3f}", (2.4, base_rate), fontsize=7.5,
               ha="right", va="bottom", color="#64748b")
ax[0].set(ylabel="fraction that needed work", ylim=(0, 1),
          title=f"{int(swapped_in.sum())} towers swapped, "
                f"{int(y[swapped_in].sum())} of them needed work")

# Condition is what moved them. If the swapped-in towers were not distinctly
# hotter on telemetry, the blend would not be the explanation for the change.
groups_plot = [("swapped IN", condition[swapped_in & np.isfinite(condition)]),
               ("swapped OUT", condition[swapped_out & np.isfinite(condition)]),
               ("everything else", condition[~(swapped_in | swapped_out) & np.isfinite(condition)])]
bp = ax[1].boxplot([g[1] for g in groups_plot], widths=0.5, showfliers=False)
# `tick_labels` on matplotlib >= 3.9, `labels` before it; set the ticks directly
# so this cell does not depend on which one the environment has.
ax[1].set_xticks(range(1, len(groups_plot) + 1),
                 [g[0] for g in groups_plot], fontsize=8.5)
ax[1].set(ylabel="condition score (one-sided, 30-day telemetry)",
          title="The second opinion is what moved them")
plt.show()

print(f"swapped in : {int(swapped_in.sum()):>3d} towers, {int(y[swapped_in].sum())} needed work "
      f"({hit_in:.3f})")
print(f"swapped out: {int(swapped_out.sum()):>3d} towers, {int(y[swapped_out].sum())} needed work "
      f"({hit_out:.3f})")
print(f"net TP      : {int(y[swapped_in].sum()) - int(y[swapped_out].sum()):+d}")
print(f"\nRead it against {base_rate:.3f}: a swap-in hit rate at the base rate means the blend "
      "is\nshuffling noise. Above it, the second opinion is finding towers the model ranked low.")

## 7. The guardrails — why the gain is believable, and where it is not

Four checks. Each one has failed silently at least once in this project's
history, which is why each is computed rather than asserted in prose.

1. **The necessary condition.** A second opinion may only act if it beats the
   model *inside the band where the decision is contested* — between the 70th
   and 90th percentile of model score. Outside that band the model is already
   confident and reordering buys nothing. `detector_auc > model_auc` in-band is
   the gate; the isolation forest fails it (a two-sided |deviation| statistic
   aimed at a monotone target), the one-sided condition read passes it.
2. **The leak check.** The ensemble must stay clearly below the oracle. Landing
   on it would mean the generator leaked, not that the policy is good.
3. **Beats the retired baseline.** The supervised path must still be worth its
   complexity against the noisy-OR index it replaced.
4. **Matched budget.** Already enforced throughout — restated here as a value.

In [ ]:
def in_band(target, model_score, evidence):
    # AUC of an evidence layer against the model's own, inside the contested
    # band — the 70th to 90th percentile of model score.
    hi, lo = np.quantile(model_score, [0.90, 0.70])
    usable = (model_score < hi) & (model_score >= lo) & np.isfinite(evidence)
    if len(set(target[usable])) < 2:
        return {"in_band_auc": None, "model_auc": None, "delta": None, "may_act": False}
    auc = float(roc_auc(target[usable], evidence[usable]))
    m = float(roc_auc(target[usable], model_score[usable]))
    return {"in_band_auc": round(auc, 4), "model_auc": round(m, 4),
            "delta": round(auc - m, 4), "may_act": auc > m}

condition_gate = in_band(y, oof, condition)
change_gate = in_band(y, oof, change)

guardrails = pd.DataFrame([
    {"check": "necessary condition (condition)",
     "value": f"{condition_gate['in_band_auc']} vs model {condition_gate['model_auc']}",
     "verdict": "PASS — may enter the ordering" if condition_gate["may_act"] else "FAIL"},
    {"check": "necessary condition (satellite change)",
     "value": f"{change_gate['in_band_auc']} vs model {change_gate['model_auc']}",
     "verdict": "FAIL — descriptive only, weight 0.0" if not change_gate["may_act"] else "PASS"},
    {"check": "leak check (PR gap to oracle)",
     "value": f"{ranking_table.loc['oracle', 'pr_auc'] - ranking_table.loc['ensemble', 'pr_auc']:+.4f}",
     "verdict": "clear" if ranking_table.loc["oracle", "pr_auc"] - ranking_table.loc["ensemble", "pr_auc"] > 0.05
                else "INVESTIGATE — too close to the generator"},
    {"check": "beats retired AHP index (PR-AUC)",
     "value": f"{ranking_table.loc['ensemble', 'pr_auc'] - ranking_table.loc['AHP index', 'pr_auc']:+.4f}",
     "verdict": "clear"},
    {"check": "matched budget", "value": f"both dispatch {budget}",
     "verdict": "enforced"},
    {"check": "labels", "value": "synthetic",
     "verdict": "figures validate the pipeline, not field accuracy"},
]).set_index("check")
display(guardrails)

assert CHANGE_BLEND_WEIGHT == 0 or change_gate["may_act"], \
    "satellite change carries dispatch weight without passing the in-band gate"

## 8. Verdict

Read this section against the tables above, not from memory — the numbers move
whenever the generator or the model is retrained, and the prose here is written
to stay true under that.

**What the comparison supports.** At a matched budget the ensemble is ahead of
the single model on F1, precision and recall, and the towers it swaps in hit well
above the base rate — so the gain is information from the telemetry layer, not a
reshuffle. Accuracy barely moves, correctly: 90% of the estate is untouched by
either policy.

**What it does not support.** The gain is small next to the gap the *data*
leaves open. Give the supervised model a 36-month telemetry aggregate and it
reaches roughly six times the F1 improvement the blend buys — and the ensemble
then loses to it. **The ensemble is a workaround for a data-plumbing gap.**
Closing the gap is worth far more than the workaround, and the blend should come
out when it lands.

**What would change with real data.** Every number here is against synthetic
labels. The first real thing to re-run is the necessary condition in §7 against
real work orders: telemetry quality is what the whole advantage rests on, and the
shipped generator's noise model is already the harsher of the two that were
measured. If `detector_auc > model_auc` fails in-band on real data, the blend
comes out regardless of what F1 says.